In [67]:
%pip install huggingface_hub pandas numpy matplotlib


Note: you may need to restart the kernel to use updated packages.


In [68]:
from huggingface_hub import hf_hub_download
import pandas as pd
import numpy as np
import os

DATASET_ID = "pedroodb/glosl-isl"

# Download only the annotation file
annotation_path = hf_hub_download(
    repo_id=DATASET_ID,
    filename="annotations.csv",
    repo_type="dataset"
)

df = pd.read_csv(annotation_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Dataset shape: (126846, 13)

Columns:
['id', 'split', 'text', 'gloss', 'gloss_type', 'signer', 'start', 'end', 'fps', 'n_frames', 'video_ref', 'pose_file', 'video_file']


,id,split,text,gloss,gloss_type,signer,start,end,fps,n_frames,video_ref,pose_file,video_file
0,1782bea75c7d-1,train,Page 111,NaN,NaN,NaN,NaN,NaN,25,113,1782bea75c7d,poses/0000/1782bea75c7d-1.pose,NaN
1,1782bea75c7d-2,train,Make it shorter.,NaN,NaN,NaN,NaN,NaN,25,104,1782bea75c7d,poses/0000/1782bea75c7d-2.pose,NaN
2,1782bea75c7d-3,train,"One day, Akbar drew a line on the floor and or...",NaN,NaN,NaN,NaN,NaN,25,196,1782bea75c7d,poses/0000/1782bea75c7d-3.pose,NaN
3,1782bea75c7d-4,train,Make this line shorter.,NaN,NaN,NaN,NaN,NaN,25,78,1782bea75c7d,poses/0000/1782bea75c7d-4.pose,NaN
4,1782bea75c7d-5,train,but don't rub out any part of it.,NaN,NaN,NaN,NaN,NaN,25,124,1782bea75c7d,poses/0013/1782bea75c7d-5.pose,NaN


In [69]:
df.isnull().sum()


id                 0
split              0
text               0
gloss         126846
gloss_type    126846
signer        126846
start         126846
end           126846
fps                0
n_frames           0
video_ref         33
pose_file          0
video_file    126846
dtype: int64

In [70]:
print("Total samples:", len(df))
print("Samples with gloss:", df["gloss"].notna().sum())
print("Samples without gloss:", df["gloss"].isna().sum())

Total samples: 126846
Samples with gloss: 0
Samples without gloss: 126846


In [71]:
df[df["gloss"].notna()][
    ["text", "gloss", "gloss_type", "signer", "n_frames"]
].head(15)

,text,gloss,gloss_type,signer,n_frames


In [72]:
sample = df.iloc[0]

print("ID:", sample["id"])
print("English text:", sample["text"])
print("Pose file:", sample["pose_file"])
print("Frames:", sample["n_frames"])
print("FPS:", sample["fps"])

ID: 1782bea75c7d-1
English text: Page 111
Pose file: poses/0000/1782bea75c7d-1.pose
Frames: 113
FPS: 25


In [73]:
from huggingface_hub import hf_hub_download
import os

pose_path = hf_hub_download(
    repo_id=DATASET_ID,
    filename=sample["pose_file"],
    repo_type="dataset"
)

print("Downloaded to:", pose_path)
print("File size:", os.path.getsize(pose_path), "bytes")

Downloaded to: C:\Users\Elza Rose Wilson\.cache\huggingface\hub\datasets--pedroodb--glosl-isl\snapshots\745795543fef509799e7e72640f286605c2ec670\poses\0000\1782bea75c7d-1.pose
File size: 995957 bytes


In [74]:
with open(pose_path, "rb") as f:
    data = f.read(200)

print(data[:200])

b'\xcd\xccL>,\x01,\x01\x00\x00\x04\x00\x0e\x00POSE_LANDMARKS\x04\x00XYZC!\x00#\x00\x01\x00\x04\x00NOSE\x0e\x00LEFT_EYE_INNER\x08\x00LEFT_EYE\x0e\x00LEFT_EYE_OUTER\x0f\x00RIGHT_EYE_INNER\t\x00RIGHT_EYE\x0f\x00RIGHT_EYE_OUTER\x08\x00LEFT_EAR\t\x00RIGHT_EAR\n\x00MOUTH_LEFT\x0b\x00MOUTH_RIGHT\r\x00LEFT_SHOULDER\x0e\x00RIGH'


In [75]:
%pip install pose-format

Note: you may need to restart the kernel to use updated packages.


In [76]:
from pose_format import Pose

with open(pose_path, "rb") as f:
    pose = Pose.read(f.read())

print(pose)

Pose
PoseHeader
Version: 0.20000000298023224
PoseHeaderDimensions(width=300, height=300, depth=0)
Bounding Box: False
Components:
PoseHeaderComponent: POSE_LANDMARKS
  Format: XYZC
  Points: ['NOSE', 'LEFT_EYE_INNER', 'LEFT_EYE', 'LEFT_EYE_OUTER', 'RIGHT_EYE_INNER', 'RIGHT_EYE', 'RIGHT_EYE_OUTER', 'LEFT_EAR', 'RIGHT_EAR', 'MOUTH_LEFT', 'MOUTH_RIGHT', 'LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_ELBOW', 'RIGHT_ELBOW', 'LEFT_WRIST', 'RIGHT_WRIST', 'LEFT_PINKY', 'RIGHT_PINKY', 'LEFT_INDEX', 'RIGHT_INDEX', 'LEFT_THUMB', 'RIGHT_THUMB', 'LEFT_HIP', 'RIGHT_HIP', 'LEFT_KNEE', 'RIGHT_KNEE', 'LEFT_ANKLE', 'RIGHT_ANKLE', 'LEFT_HEEL', 'RIGHT_HEEL', 'LEFT_FOOT_INDEX', 'RIGHT_FOOT_INDEX']
  Limbs: 35
  Colors: 1

PoseHeaderComponent: FACE_LANDMARKS
  Format: XYZC
  Points: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40

In [77]:
print("Header:")
print(pose.header)

print("\nBody:")
print(pose.body)

Header:
PoseHeader
Version: 0.20000000298023224
PoseHeaderDimensions(width=300, height=300, depth=0)
Bounding Box: False
Components:
PoseHeaderComponent: POSE_LANDMARKS
  Format: XYZC
  Points: ['NOSE', 'LEFT_EYE_INNER', 'LEFT_EYE', 'LEFT_EYE_OUTER', 'RIGHT_EYE_INNER', 'RIGHT_EYE', 'RIGHT_EYE_OUTER', 'LEFT_EAR', 'RIGHT_EAR', 'MOUTH_LEFT', 'MOUTH_RIGHT', 'LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_ELBOW', 'RIGHT_ELBOW', 'LEFT_WRIST', 'RIGHT_WRIST', 'LEFT_PINKY', 'RIGHT_PINKY', 'LEFT_INDEX', 'RIGHT_INDEX', 'LEFT_THUMB', 'RIGHT_THUMB', 'LEFT_HIP', 'RIGHT_HIP', 'LEFT_KNEE', 'RIGHT_KNEE', 'LEFT_ANKLE', 'RIGHT_ANKLE', 'LEFT_HEEL', 'RIGHT_HEEL', 'LEFT_FOOT_INDEX', 'RIGHT_FOOT_INDEX']
  Limbs: 35
  Colors: 1

PoseHeaderComponent: FACE_LANDMARKS
  Format: XYZC
  Points: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', 

In [78]:
data = pose.body.data
confidence = pose.body.confidence

body_landmarks = data[:, :, 0:33, :]
face_landmarks = data[:, :, 33:501, :]
left_hand_landmarks = data[:, :, 501:522, :]
right_hand_landmarks = data[:, :, 522:543, :]

print("Body:", body_landmarks.shape)
print("Face:", face_landmarks.shape)
print("Left hand:", left_hand_landmarks.shape)
print("Right hand:", right_hand_landmarks.shape)

Body: (113, 1, 33, 3)
Face: (113, 1, 468, 3)
Left hand: (113, 1, 21, 3)
Right hand: (113, 1, 21, 3)


In [79]:
print("Coordinate sample:")
print(data[0, 0, :5, :])

print("\nConfidence sample:")
print(confidence[0, 0, :10])

print("\nMinimum confidence:", confidence.min())
print("Maximum confidence:", confidence.max())

Coordinate sample:
[[151.30836486816406 79.6260986328125 -1.3490707874298096]
 [159.58148193359375 67.36503601074219 -1.2962653636932373]
 [163.0798797607422 67.51565551757812 -1.2962437868118286]
 [167.6418914794922 67.33982849121094 -1.2959084510803223]
 [144.421875 68.04610443115234 -1.294122338294983]]

Confidence sample:
[0.99996436 0.9999467  0.99993765 0.9999236  0.9999577  0.99996114
 0.99996555 0.9999287  0.99996924 0.9999907 ]

Minimum confidence: 0.0
Maximum confidence: 1.0


In [80]:
total_landmarks = confidence.size

missing_landmarks = np.sum(confidence == 0)

missing_percentage = (missing_landmarks / total_landmarks) * 100

print("Total landmarks:", total_landmarks)
print("Missing landmarks:", missing_landmarks)
print("Missing percentage:", missing_percentage)

Total landmarks: 61359
Missing landmarks: 378
Missing percentage: 0.6160465457390114


In [81]:
parts = {
    "Body": confidence[:, :, 0:33],
    "Face": confidence[:, :, 33:501],
    "Left hand": confidence[:, :, 501:522],
    "Right hand": confidence[:, :, 522:543]
}

for name, part in parts.items():
    missing = np.sum(part == 0)
    total = part.size

    print(
        name,
        "Missing:", missing,
        "Percentage:", round((missing / total) * 100, 2), "%"
    )

Body Missing: 0 Percentage: 0.0 %
Face Missing: 0 Percentage: 0.0 %
Left hand Missing: 147 Percentage: 6.19 %
Right hand Missing: 231 Percentage: 9.73 %


In [82]:
print("X min:", data[:, :, :, 0].min())
print("X max:", data[:, :, :, 0].max())

print("Y min:", data[:, :, :, 1].min())
print("Y max:", data[:, :, :, 1].max())

print("Z min:", data[:, :, :, 2].min())
print("Z max:", data[:, :, :, 2].max())

X min: 4.6108637
X max: 255.6591
Y min: 43.423927
Y max: 579.1429
Z min: -2.207844
Z max: 1.1595968


In [83]:
print(pose.header)

PoseHeader
Version: 0.20000000298023224
PoseHeaderDimensions(width=300, height=300, depth=0)
Bounding Box: False
Components:
PoseHeaderComponent: POSE_LANDMARKS
  Format: XYZC
  Points: ['NOSE', 'LEFT_EYE_INNER', 'LEFT_EYE', 'LEFT_EYE_OUTER', 'RIGHT_EYE_INNER', 'RIGHT_EYE', 'RIGHT_EYE_OUTER', 'LEFT_EAR', 'RIGHT_EAR', 'MOUTH_LEFT', 'MOUTH_RIGHT', 'LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_ELBOW', 'RIGHT_ELBOW', 'LEFT_WRIST', 'RIGHT_WRIST', 'LEFT_PINKY', 'RIGHT_PINKY', 'LEFT_INDEX', 'RIGHT_INDEX', 'LEFT_THUMB', 'RIGHT_THUMB', 'LEFT_HIP', 'RIGHT_HIP', 'LEFT_KNEE', 'RIGHT_KNEE', 'LEFT_ANKLE', 'RIGHT_ANKLE', 'LEFT_HEEL', 'RIGHT_HEEL', 'LEFT_FOOT_INDEX', 'RIGHT_FOOT_INDEX']
  Limbs: 35
  Colors: 1

PoseHeaderComponent: FACE_LANDMARKS
  Format: XYZC
  Points: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '4

In [84]:
%pip install glosl

Note: you may need to restart the kernel to use updated packages.


In [85]:
import glosl

print("GloSL imported successfully!")

GloSL imported successfully!


In [86]:
import glosl

print("GloSL imported successfully!")
print(glosl.__file__)

GloSL imported successfully!
c:\Users\Elza Rose Wilson\personalworks\isl-cv-project\.venv\Lib\site-packages\glosl\__init__.py


In [87]:
print(dir(glosl))


['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'audit', 'check', 'corpora', 'dataset', 'evaluate', 'format', 'hub', 'load', 'splits', 'validate']


In [88]:
print(type(glosl.load))
print(dir(glosl.load))

<class 'function'>
['__annotate__', '__annotations__', '__builtins__', '__call__', '__class__', '__closure__', '__code__', '__defaults__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__get__', '__getattribute__', '__getstate__', '__globals__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__kwdefaults__', '__le__', '__lt__', '__module__', '__name__', '__ne__', '__new__', '__qualname__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__type_params__']


In [89]:
help(glosl.load)

Help on function load in module glosl:

load(source: str, revision: str | None = None)



In [90]:
print(dir(glosl.corpora))

['__annotate__', '__annotations__', '__builtins__', '__call__', '__class__', '__closure__', '__code__', '__defaults__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__get__', '__getattribute__', '__getstate__', '__globals__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__kwdefaults__', '__le__', '__lt__', '__module__', '__name__', '__ne__', '__new__', '__qualname__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__type_params__']


In [91]:
help(glosl.corpora)

Help on function corpora in module glosl:

corpora(version: str | None = None) -> list[dict]



In [92]:
print(glosl.load.__doc__)

None


In [93]:
available_corpora = glosl.corpora()

print("Available datasets:")

for corpus in available_corpora:
    print(corpus)

RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-6b00aa10-2f793435378d17e22102fc74;4f437a79-6953-4966-b249-095ee98801d1)

Repository Not Found for url: https://huggingface.co/api/datasets/pedroodb/glosl-registry/revision/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.

In [ ]:
ds = glosl.load("pedroodb/glosl-isl")

print(ds)

In [ ]:
try:
    ds = glosl.load("pedroodb/glosl-isl")
    print("Dataset loaded successfully!")
    print(ds)
except Exception as e:
    print("Loading failed:")
    print(type(e).__name__)
    print(str(e))

Dataset loaded successfully!


In [ ]:
print(type(ds))
print(dir(ds))

<class 'glosl.dataset.Dataset'>
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_by_id', '_file', 'annotations', 'card', 'download', 'lang', 'local', 'pose', 'revision', 'root', 'source', 'split']


In [ ]:
help(ds)


Help on Dataset in module glosl.dataset object:

class Dataset(builtins.object)
 |  Dataset(source: str, revision: str | None = None)
 |
 |  Methods defined here:
 |
 |  __init__(self, source: str, revision: str | None = None)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  download(self, split: str | None = None, workers: int = 16) -> Path
 |      Fetch every pose (of one split) at once, cached and resumable; returns the local root.
 |
 |  pose(self, clip_id: str)
 |
 |  split(self, name: str) -> pd.DataFrame
 |
 |  ----------------------------------------------------------------------
 |  Readonly properties defined here:
 |
 |  lang
 |
 |  ----------------------------------------------------------------------
 |  Data descriptors defined here:
 |
 |  __dict__
 |      dictionary for instance variables
 |
 |  __weakref__
 |      list of weak references to the object



In [ ]:
train_df = ds.split("train")
dev_df = ds.split("dev")
test_df = ds.split("test")

print("Training samples:", len(train_df))
print("Validation samples:", len(dev_df))
print("Testing samples:", len(test_df))

print("\nAvailable columns:")
print(train_df.columns.tolist())

print("\nFirst 5 training samples:")
print(train_df[["id", "text", "n_frames", "fps"]].head())

Training samples: 101481
Validation samples: 12683
Testing samples: 12682

Available columns:
['id', 'split', 'text', 'gloss', 'gloss_type', 'signer', 'start', 'end', 'fps', 'n_frames', 'video_ref', 'pose_file', 'video_file', 'seen_text']

First 5 training samples:
               id                                               text n_frames  \
0  1782bea75c7d-1                                           Page 111      113   
1  1782bea75c7d-2                                   Make it shorter.      104   
2  1782bea75c7d-3  One day, Akbar drew a line on the floor and or...      196   
3  1782bea75c7d-4                            Make this line shorter.       78   
4  1782bea75c7d-5                  but don't rub out any part of it.      124   

  fps  
0  25  
1  25  
2  25  
3  25  
4  25  


In [ ]:
sample = train_df.iloc[0]

sample_id = sample["id"]

sample_pose = ds.pose(sample_id)

print("Sample ID:", sample_id)
print("Translation:", sample["text"])
print("Pose type:", type(sample_pose))
print("Pose data shape:", sample_pose.body.data.shape)

Sample ID: 1782bea75c7d-1
Translation: Page 111
Pose type: <class 'pose_format.pose.Pose'>
Pose data shape: (113, 1, 543, 3)


In [ ]:
sample_data = sample_pose.body.data

print("First 5 landmarks:")
print(sample_data[0, 0, :5, :])

print("\nCoordinate ranges:")

for i, axis in enumerate(["X", "Y", "Z"]):
    print(
        axis,
        "Min:", sample_data[:, :, :, i].min(),
        "Max:", sample_data[:, :, :, i].max()
    )

First 5 landmarks:
[[151.30836486816406 79.6260986328125 -1.3490707874298096]
 [159.58148193359375 67.36503601074219 -1.2962653636932373]
 [163.0798797607422 67.51565551757812 -1.2962437868118286]
 [167.6418914794922 67.33982849121094 -1.2959084510803223]
 [144.421875 68.04610443115234 -1.294122338294983]]

Coordinate ranges:
X Min: 4.6108637 Max: 255.6591
Y Min: 43.423927 Max: 579.1429
Z Min: -2.207844 Max: 1.1595968


In [ ]:
import pkgutil
import glosl

for module in pkgutil.walk_packages(
    glosl.__path__,
    prefix="glosl."
):
    print(module.name)

glosl.audit
glosl.check
glosl.cli
glosl.dataset
glosl.extract
glosl.format
glosl.hub
glosl.keypoints
glosl.pytorch
glosl.score
glosl.splits
glosl.validate


In [ ]:
import sys
print(sys.version)

3.14.5 (tags/v3.14.5:5607950, May 10 2026, 10:43:50) [MSC v.1944 64 bit (AMD64)]


In [ ]:
import inspect
import glosl.pytorch

print(inspect.getsource(glosl.pytorch))

"""PyTorch datasets over GloSL corpora on the Hugging Face Hub (or local folders), and the multilingual training mix.

    from torch.utils.data import DataLoader
    from glosl.pytorch import PoseDataset, Mix, BalancedBatchSampler, collate

    test = PoseDataset("pedroodb/glosl-phoenix14t", "test")
    loader = DataLoader(test, batch_size=32, collate_fn=collate)

    train = Mix([PoseDataset(repo, "train", layout="unisign69") for repo in repos])
    sampler = BalancedBatchSampler(train, batch_size=32)       # the GloSL paper's mix
    loader = DataLoader(train, batch_sampler=sampler, collate_fn=collate, num_workers=4)
    for epoch in range(epochs):
        sampler.set_epoch(epoch)
        for batch in loader:
            batch["pose"], batch["mask"], batch["text"]          # [B, T, V, 3], [B, T], list of strings

An item is a dict: "pose" float32 [T, V, 3] (x / frame width, y / frame height, confidence; missing points 0, see
glosl.keypoints), "text", "gloss", "id", "corpus", "lang" 

In [ ]:
from glosl.keypoints import read_pose, to_layout
pose_path = ds._file(sample["pose_file"])

xy, conf, width, height, fps = read_pose(pose_path)

processed = to_layout(
    xy, conf, width, height,
    layout="holistic"
)

print("Original XY:", xy.shape)
print("Confidence:", conf.shape)
print("Processed:", processed.shape)
print("First 5 landmarks:", processed[0, :5])

Original XY: (113, 543, 2)
Confidence: (113, 543)
Processed: (113, 543, 3)
First 5 landmarks: [[0.5043612  0.26542032 0.99996436]
 [0.53193825 0.22455011 0.9999467 ]
 [0.5435996  0.22505218 0.99993765]
 [0.5588063  0.2244661  0.9999236 ]
 [0.48140624 0.22682035 0.9999577 ]]


In [ ]:
import inspect
import glosl.pytorch

print(inspect.getsource(glosl.pytorch.PoseDataset))


class PoseDataset(Dataset):
    """One split of one GloSL corpus. `source` is a Hub dataset repo id or a local folder.

    download=True fetches the split's poses once (resumable, cached by huggingface_hub); False reads each pose on
    first use. max_frames keeps at most that many frames, evenly spaced. unseen_only keeps the dev or test clips whose
    sentence does not occur in training (diagnostics/clip_tags.csv). transform(item) -> item runs last."""

    def __init__(self, source: str, split: str = "train", layout="holistic", revision: str | None = None,
                 download: bool = True, max_frames: int | None = None, unseen_only: bool = False, transform=None):
        self.ds = load(source, revision)
        rows = self.ds.split(split)
        if unseen_only:
            if "seen_text" not in rows:
                raise ValueError(f"{source} has no diagnostics/clip_tags.csv: run glosl check")
            rows = rows[rows["seen_text"] == "0"].reset_index(drop=True)
        

In [ ]:
source = inspect.getsource(glosl.pytorch.PoseDataset)

print(source[source.find("def __getitem__"):])

def __getitem__(self, i: int) -> dict:
        r = self.rows.iloc[i]
        pose = to_layout(*read_pose(self.path(i))[:4], layout=self.layout)
        if self.max_frames and len(pose) > self.max_frames:
            pose = pose[np.linspace(0, len(pose) - 1, self.max_frames).round().astype(int)]
        item = {"pose": torch.from_numpy(pose), "text": r["text"], "gloss": r["gloss"], "id": r["id"],
                "corpus": self.corpus, "lang": self.lang, "sign_language": self.sign_language}
        return self.transform(item) if self.transform else item



In [ ]:
from glosl.pytorch import PoseDataset

train_data = PoseDataset(
    source="pedroodb/glosl-isl",
    split="train",
    layout="holistic",
    download=False,
    max_frames=150
)

print("Dataset size:", len(train_data))
print("Total hours:", train_data.hours)

Dataset size: 101481
Total hours: 215.37828856960408


In [ ]:
item = train_data[0]

print("Keys:", item.keys())
print("Pose shape:", item["pose"].shape)
print("Pose dtype:", item["pose"].dtype)
print("Translation:", item["text"])
print("Sample ID:", item["id"])

Keys: dict_keys(['pose', 'text', 'gloss', 'id', 'corpus', 'lang', 'sign_language'])
Pose shape: torch.Size([113, 543, 3])
Pose dtype: torch.float32
Translation: Page 111
Sample ID: 1782bea75c7d-1


In [ ]:
import inspect
print(inspect.getsource(glosl.pytorch.collate))

def collate(items: list[dict]) -> dict:
    """Pad poses to the longest clip: pose [B, T, V, 3], mask [B, T] (True on real frames), lengths [B]; the other
    fields as lists."""
    lengths = torch.tensor([len(x["pose"]) for x in items])
    pose = torch.zeros((len(items), int(lengths.max()), *items[0]["pose"].shape[1:]), dtype=items[0]["pose"].dtype)
    for b, x in enumerate(items):
        pose[b, :len(x["pose"])] = x["pose"]
    out = {"pose": pose, "mask": torch.arange(pose.shape[1])[None] < lengths[:, None], "lengths": lengths}
    out.update({k: [x[k] for x in items] for k in items[0] if k != "pose"})
    return out



In [ ]:
from glosl.pytorch import collate

items = [train_data[i] for i in range(3)]

batch = collate(items)

print("Pose:", batch["pose"].shape)
print("Mask:", batch["mask"].shape)
print("Lengths:", batch["lengths"])
print("Texts:", batch["text"])

poses/0000/1782bea75c7d-2.pose: reconstructing file:   0%|          |  0.00B /  918kB            

poses/0000/1782bea75c7d-2.pose: downloading bytes:           |  0.00B            

poses/0000/1782bea75c7d-3.pose: reconstructing file:   0%|          |  0.00B / 1.72MB            

poses/0000/1782bea75c7d-3.pose: downloading bytes:           |  0.00B            

Pose: torch.Size([3, 150, 543, 3])
Mask: torch.Size([3, 150])
Lengths: tensor([113, 104, 150])
Texts: ['Page 111', 'Make it shorter.', 'One day, Akbar drew a line on the floor and ordered.']


In [ ]:
import glosl

ds = glosl.load("pedroodb/glosl-isl")

print(ds)

In [ ]:

def tokenization(i):
    punctuations = [".", ",", "?", "!", ";", ":"]
    i = i.lower()

    for j in punctuations:
        i = i.replace(j, " " + j)

    tokens = i.split()
    return tokens


def create_vocab(tokens):
    vocab_list = list(dict.fromkeys(tokens))

    vocab = {
        "<PAD>": 0,
        "<UNK>": 1,
        "<SOS>": 2,
        "<EOS>": 3
    }

    for j, token in enumerate(vocab_list, start=4):
        vocab[token] = j

    return vocab


def text_to_sequence(s, vocab):
    tokens = tokenization(s)
    sequence = [vocab["<SOS>"]]

    for i in tokens:
        if i in vocab:
            sequence.append(vocab[i])
        else:
            sequence.append(vocab["<UNK>"])

    sequence.append(vocab["<EOS>"])
    return sequence


def padding(list_s):
    n = max(len(i) for i in list_s)

    for i in list_s:
        if len(i) < n:
            i.extend([0] * (n - len(i)))

    return list_s

def create_mask(batch):
    masks = []

    for sequence in batch:
        mask = []

        for i in sequence:
            if i != 0:
                mask.append(1)
            else:
                mask.append(0)

        masks.append(mask)

    return masks 

# LOAD TRAINING DATA
train_data = ds.split("train")
sentences = train_data["text"].to_list()


# TOKENIZATION OF TRAINING DATA
all_tokens = []

for s in sentences:
    tokens = tokenization(s)
    all_tokens.extend(tokens)


# CREATE VOCABULARY
vocab = create_vocab(all_tokens)


# CONVERT ALL SENTENCES INTO NUMERICAL SEQUENCES
all_sequences = []

for s in sentences:
    seq = text_to_sequence(s, vocab)
    all_sequences.append(seq)


# CREATE BATCHES AND APPLY PADDING
batch_size = 32
padded_batches = []

for i in range(0, len(all_sequences), batch_size):
    batch = all_sequences[i:i + batch_size]
    padded_batch = padding(batch)
    padded_batches.append(padded_batch)
mask = create_mask(padded_batches[0])

# CHECK RESULTS
print("Total sentences:", len(sentences))
print("Total tokens:", len(all_tokens))
print("Vocabulary size:", len(vocab))
print("Total sequences:", len(all_sequences))
print("Total batches:", len(padded_batches))

print("\nFirst sequence:", all_sequences[0])
print("\nFirst padded batch:", padded_batches[0])
print("First sequence:", padded_batches[0][0])
print("Its mask:", mask[0])

Total sentences: 101481
Total tokens: 1223992
Vocabulary size: 39084
Total sequences: 101481
Total batches: 3172

First sequence: [2, 4, 5, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

First padded batch: [[2, 4, 5, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 6, 7, 8, 9, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 9, 3, 0, 0, 0], [2, 6, 22, 16, 8, 9, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 23, 24, 25, 26, 27, 28, 29, 7, 9, 3, 0, 0, 0, 0, 0, 0, 0], [2, 30, 10, 31, 32, 33, 34, 9, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 35, 36, 37, 38, 18, 16, 20, 39, 40, 9, 3, 0, 0, 0, 0, 0, 0], [2, 30, 10, 41, 42, 29, 27, 43, 33, 6, 7, 44, 9, 3, 0, 0, 0, 0], [2, 30, 10, 41, 42, 29, 45, 7, 41, 46, 47, 8, 48, 49, 7, 9, 3, 0], [2, 50, 51, 52, 9, 53, 7, 39, 54, 55, 12, 56, 57, 58, 18, 16, 9, 3], [2, 59, 60, 61, 62, 32, 50, 63, 34, 9, 3, 0, 0, 0, 0, 0, 0, 0], [2, 64, 65, 66, 9, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 50, 14, 15, 44, 16, 9, 3, 0, 0, 0, 0,

In [ ]:
del list

In [ ]:
def padding(list_s):
    n = max(len(i) for i in list_s)

    for i in list_s:
        if len(i) < n:
            i.extend([0] * (n - len(i)))

    return list_s

a = [2, 5, 3]
b = [2, 4, 6, 7, 3]
c = [2, 8, 3] 
print(padding([a, b, c]))

[[2, 5, 3, 0, 0], [2, 4, 6, 7, 3], [2, 8, 3, 0, 0]]


In [ ]:
import torch

print(torch.cuda.is_available())

False


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

PyTorch version: 2.15.0+cpu
CUDA available: False
GPU count: 0


In [ ]:
import sys
print(sys.executable)

c:\Users\Elza Rose Wilson\personalworks\isl-cv-project\.venv\Scripts\python.exe


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")

PyTorch: 2.15.0+cpu
CUDA available: False


In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.15.0+cpu
False


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
import sys
print(sys.executable)

import torch
print(torch.__file__)

c:\Users\Elza Rose Wilson\personalworks\isl-cv-project\.venv\Scripts\python.exe
c:\Users\Elza Rose Wilson\personalworks\isl-cv-project\.venv\Lib\site-packages\torch\__init__.py


In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cu128
True


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x = torch.rand(3, 3).to(device)

print("Device:", x.device)
print(x)

Device: cuda:0
tensor([[0.0933, 0.5319, 0.4376],
        [0.7972, 0.9050, 0.8340],
        [0.6307, 0.4710, 0.2669]], device='cuda:0')


In [ ]:
print("ds" in globals())

False


In [ ]:
train_data = ds.split("train")

print(type(train_data))
print(train_data.iloc[0])

<class 'pandas.DataFrame'>
id                            1782bea75c7d-1
split                                  train
text                                Page 111
gloss                                       
gloss_type                                  
signer                                      
start                                       
end                                         
fps                                       25
n_frames                                 113
video_ref                       1782bea75c7d
pose_file     poses/0000/1782bea75c7d-1.pose
video_file                                  
seen_text                                NaN
Name: 0, dtype: str


In [94]:
from glosl.pytorch import PoseDataset

pose_ds=PoseDataset(
  source="pedroodb/glosl-isl",
  split="train",
  layout="holistic",
  download=False
)
sample= pose_ds[0]

print("Available keys:", sample.keys())
print("Pose shape:", sample["pose"].shape)
print("Pose dtype:", sample["pose"].dtype)
print("Text:", sample["text"])

Available keys: dict_keys(['pose', 'text', 'gloss', 'id', 'corpus', 'lang', 'sign_language'])
Pose shape: torch.Size([113, 543, 3])
Pose dtype: torch.float32
Text: Page 111


In [ ]:
for i in range(10):
    sample = pose_ds[i]

    print(f"\nSample {i}")
    print("ID:", sample["id"])
    print("Text:", sample["text"])
    print("Pose shape:", sample["pose"].shape)


Sample 0
ID: 1782bea75c7d-1
Text: Page 111
Pose shape: torch.Size([113, 543, 3])

Sample 1
ID: 1782bea75c7d-2
Text: Make it shorter.
Pose shape: torch.Size([104, 543, 3])

Sample 2
ID: 1782bea75c7d-3
Text: One day, Akbar drew a line on the floor and ordered.
Pose shape: torch.Size([196, 543, 3])


c:\Users\Elza Rose Wilson\personalworks\isl-cv-project\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Elza Rose Wilson\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


poses/0000/1782bea75c7d-4.pose: reconstructing file:   0%|          |  0.00B /  692kB            

poses/0000/1782bea75c7d-4.pose: downloading bytes:           |  0.00B            

c:\Users\Elza Rose Wilson\personalworks\isl-cv-project\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Elza Rose Wilson\.cache\huggingface\hub\datasets--pedroodb--glosl-isl. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)



Sample 3
ID: 1782bea75c7d-4
Text: Make this line shorter.
Pose shape: torch.Size([78, 543, 3])


... (14 outputs left out)


Sample 8
ID: 1782bea75c7d-9
Text: No one could think of how it could be made shorter without erasing it.
Pose shape: torch.Size([174, 543, 3])


poses/0013/1782bea75c7d-10.pose: reconstructing file:   0%|          |  0.00B / 1.72MB            

poses/0013/1782bea75c7d-10.pose: downloading bytes:           |  0.00B            


Sample 9
ID: 1782bea75c7d-10
Text: Birbal started smiling. When it was his turn, he went near the line.
Pose shape: torch.Size([196, 543, 3])


In [ ]:
import re

texts = train_data["text"].dropna().tolist()

def is_useful(text):
    text = str(text).strip()

    if len(text.split()) < 2:
        return False

    if re.fullmatch(r"page\s+\d+", text.lower()):
        return False

    return True

useful_texts = [text for text in texts if is_useful(text)]

print("Total annotations:", len(texts))
print("Useful annotations:", len(useful_texts))
print("Filtered annotations:", len(texts) - len(useful_texts))
print("Useful percentage:", round(len(useful_texts) / len(texts) * 100, 2))

Total annotations: 101481
Useful annotations: 98195
Filtered annotations: 3286
Useful percentage: 96.76


In [ ]:
print("Dataset length:", len(pose_ds))
print("First sample ID:", pose_ds[0]["id"])
print("First sample text:", pose_ds[0]["text"])

Dataset length: 101481
First sample ID: 1782bea75c7d-1
First sample text: Page 111


In [ ]:
print(train_data[["id", "text", "pose_file"]].head())

               id                                               text  \
0  1782bea75c7d-1                                           Page 111   
1  1782bea75c7d-2                                   Make it shorter.   
2  1782bea75c7d-3  One day, Akbar drew a line on the floor and or...   
3  1782bea75c7d-4                            Make this line shorter.   
4  1782bea75c7d-5                  but don't rub out any part of it.   

                        pose_file  
0  poses/0000/1782bea75c7d-1.pose  
1  poses/0000/1782bea75c7d-2.pose  
2  poses/0000/1782bea75c7d-3.pose  
3  poses/0000/1782bea75c7d-4.pose  
4  poses/0013/1782bea75c7d-5.pose  


In [ ]:
print("Annotation rows:", len(train_data))
print("Pose dataset:", len(pose_ds))

for i in range(5):
    annotation_id = train_data.iloc[i]["id"]
    pose_id = pose_ds[i]["id"]

    print(
        i,
        "Annotation ID:", annotation_id,
        "| Pose ID:", pose_id,
        "| Match:", annotation_id == pose_id
    )

Annotation rows: 101481
Pose dataset: 101481
0 Annotation ID: 1782bea75c7d-1 | Pose ID: 1782bea75c7d-1 | Match: True
1 Annotation ID: 1782bea75c7d-2 | Pose ID: 1782bea75c7d-2 | Match: True
2 Annotation ID: 1782bea75c7d-3 | Pose ID: 1782bea75c7d-3 | Match: True
3 Annotation ID: 1782bea75c7d-4 | Pose ID: 1782bea75c7d-4 | Match: True
4 Annotation ID: 1782bea75c7d-5 | Pose ID: 1782bea75c7d-5 | Match: True


In [95]:

import re
import torch
from torch.utils.data import Dataset

# 1. Filter annotations and preserve original row indices
valid_indices = []

for i, text in enumerate(train_data["text"]):

    text = str(text).strip()

    if len(text.split()) >= 2 and not re.fullmatch(
        r"page\s+\d+", text.lower()
    ):
        valid_indices.append(i)

print("Valid samples:", len(valid_indices))


# 2. Create our custom PyTorch Dataset
class ISLDataset(Dataset):

    def __init__(self, annotations, pose_dataset, valid_indices, vocab):

        self.annotations = annotations
        self.pose_dataset = pose_dataset
        self.valid_indices = valid_indices
        self.vocab = vocab

    def __len__(self):

        return len(self.valid_indices)

    def __getitem__(self, index):

        # Retrieve original row index
        original_index = self.valid_indices[index]

        # Load corresponding pose sample
        sample = self.pose_dataset[original_index]

        pose = sample["pose"]
        text = sample["text"]

        # Convert English sentence into token IDs
        sequence = text_to_sequence(text, self.vocab)

        # Convert token IDs into PyTorch tensor
        sequence = torch.tensor(sequence, dtype=torch.long)

        return pose, sequence


# 3. Create the training Dataset object
train_dataset = ISLDataset(
    annotations=train_data,
    pose_dataset=pose_ds,
    valid_indices=valid_indices,
    vocab=vocab
)


# 4. Test the Dataset
print("\nDataset size:", len(train_dataset))

pose, target = train_dataset[0]

print("\nFirst training example:")
print("Pose shape:", pose.shape)
print("Pose dtype:", pose.dtype)
print("Target:", target)
print("Target shape:", target.shape)


Valid samples: 98195

Dataset size: 98195

First training example:
Pose shape: torch.Size([104, 543, 3])
Pose dtype: torch.float32
Target: tensor([2, 6, 7, 8, 9, 3])
Target shape: torch.Size([6])


In [96]:

def collate_fn(batch):

    # Separate poses and text sequences
    poses = [item[0] for item in batch]
    texts = [item[1] for item in batch]

    # Store original lengths
    pose_lengths = torch.tensor(
        [pose.shape[0] for pose in poses],
        dtype=torch.long
    )

    text_lengths = torch.tensor(
        [text.shape[0] for text in texts],
        dtype=torch.long
    )

    # Find maximum lengths in this batch
    max_frames = pose_lengths.max().item()
    max_tokens = text_lengths.max().item()

    # Get landmark dimensions
    num_landmarks = poses[0].shape[1]
    num_features = poses[0].shape[2]

    # Create zero-filled padded tensors
    padded_poses = torch.zeros(
        len(batch),
        max_frames,
        num_landmarks,
        num_features,
        dtype=torch.float32
    )

    padded_texts = torch.zeros(
        len(batch),
        max_tokens,
        dtype=torch.long
    )

    # Copy original sequences into padded tensors
    for i in range(len(batch)):

        padded_poses[i, :pose_lengths[i]] = poses[i]

        padded_texts[i, :text_lengths[i]] = texts[i]

    return padded_poses, padded_texts, pose_lengths, text_lengths
from torch.utils.data import DataLoader

batch_size = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)
poses, texts, pose_lengths, text_lengths = next(iter(train_loader))

print("Pose batch shape:", poses.shape)
print("Text batch shape:", texts.shape)

print("Pose lengths:", pose_lengths)
print("Text lengths:", text_lengths)

print("Pose dtype:", poses.dtype)
print("Text dtype:", texts.dtype)

c:\Users\Elza Rose Wilson\personalworks\isl-cv-project\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Elza Rose Wilson\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


poses/0016/bkIHDsgZ0RA--16.pose: reconstructing file:   0%|          |  0.00B /  501kB            

poses/0016/bkIHDsgZ0RA--16.pose: downloading bytes:           |  0.00B            

c:\Users\Elza Rose Wilson\personalworks\isl-cv-project\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Elza Rose Wilson\.cache\huggingface\hub\datasets--pedroodb--glosl-isl. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


poses/0005/RvdDj-FcFkQ--18.pose: reconstructing file:   0%|          |  0.00B / 2.13MB            

poses/0005/RvdDj-FcFkQ--18.pose: downloading bytes:           |  0.00B            

... (25 outputs left out)

poses/0015/PfZguvoX6qI--137.pose: downloading bytes:           |  0.00B            

poses/0004/J-nRkkZEXmc--14.pose: reconstructing file:   0%|          |  0.00B /  874kB            

poses/0004/J-nRkkZEXmc--14.pose: downloading bytes:           |  0.00B            

Pose batch shape: torch.Size([16, 345, 543, 3])
Text batch shape: torch.Size([16, 24])
Pose lengths: tensor([ 56, 244, 238, 309,  97, 345,  68, 177, 165,  88, 106, 199, 161, 298,
         74,  99])
Text lengths: tensor([ 7, 20, 18, 20, 15, 24, 15, 16, 17,  8, 10, 14, 14,  5,  9,  9])
Pose dtype: torch.float32
Text dtype: torch.int64


In [ ]:
print("Vocabulary:", len(vocab))
print("Pose dataset:", len(pose_ds))
print("Training dataset:", len(train_dataset))
print("Training batches:", len(train_loader))
print("Model class:", ISLTranslator)

In [ ]:

import torch
from torch.utils.data import DataLoader

# Clear unused GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()

batch_size = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=False
)

print("DataLoader created successfully")


DataLoader created successfully


In [ ]:

poses, texts, pose_lengths, text_lengths = next(iter(train_loader))

print("Pose batch shape:", poses.shape)
print("Text batch shape:", texts.shape)

print("Pose lengths:", pose_lengths)
print("Text lengths:", text_lengths)

print("Pose device:", poses.device)
print("Text device:", texts.device)


poses/0011/yWHPfJPfhYM--9.pose: reconstructing file:   0%|          |  0.00B / 4.25MB            

poses/0011/yWHPfJPfhYM--9.pose: downloading bytes:           |  0.00B            

poses/0011/vbuEvtq_3Fg--3.pose: reconstructing file:   0%|          |  0.00B / 2.27MB            

poses/0011/vbuEvtq_3Fg--3.pose: downloading bytes:           |  0.00B            

Pose batch shape: torch.Size([2, 488, 543, 3])
Text batch shape: torch.Size([2, 28])
Pose lengths: tensor([488, 260])
Text lengths: tensor([18, 28])
Pose device: cpu
Text device: cpu


In [ ]:
import time

start = time.time()

for i, batch in enumerate(train_loader):
    if i == 4:
        break

end = time.time()

print("Time for 5 batches:", round(end - start, 2), "seconds")

poses/0002/CDnCd4RlVIw--21.pose: reconstructing file:   0%|          |  0.00B / 1.36MB            

poses/0002/CDnCd4RlVIw--21.pose: downloading bytes:           |  0.00B            

poses/0002/C8xfqZNQd70--78.pose: reconstructing file:   0%|          |  0.00B /  675kB            

poses/0002/C8xfqZNQd70--78.pose: downloading bytes:           |  0.00B            

poses/0004/Ns4N7Q-mkGQ--11.pose: reconstructing file:   0%|          |  0.00B / 2.58MB            

poses/0004/Ns4N7Q-mkGQ--11.pose: downloading bytes:           |  0.00B            

... (11 outputs left out)

poses/0002/C6yPDcZh7Rc--4.pose: downloading bytes:           |  0.00B            

poses/0006/WkXIBPerf_Y--22.pose: reconstructing file:   0%|          |  0.00B /  396kB            

poses/0006/WkXIBPerf_Y--22.pose: downloading bytes:           |  0.00B            

Time for 5 batches: 37.49 seconds


In [ ]:
import time

# First access
start = time.time()
sample = train_dataset[0]
print("First access:", round(time.time() - start, 2), "seconds")

# Second access to the same sample
start = time.time()
sample = train_dataset[0]
print("Second access:", round(time.time() - start, 2), "seconds")

First access: 0.44 seconds
Second access: 0.26 seconds


In [ ]:
from torch.utils.data import Subset, DataLoader

subset_size = 2000

train_subset = Subset(
    train_dataset,
    range(subset_size)
)

train_loader = DataLoader(
    train_subset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=False
)

print("Training samples:", len(train_subset))
print("Batches per epoch:", len(train_loader))

Training samples: 2000
Batches per epoch: 1000


In [ ]:

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


class ISLTranslator(nn.Module):

    def __init__(self, vocab_size, hidden_size=128, embedding_dim=128):
        super().__init__()

        self.hidden_size = hidden_size

        # Convert each frame's 543 landmarks x 3 features
        # into a smaller feature representation.
        self.input_projection = nn.Sequential(
            nn.Linear(543 * 3, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # Bidirectional encoder
        self.encoder = nn.LSTM(
            input_size=256,
            hidden_size=hidden_size,
            batch_first=True,
            bidirectional=True
        )

        # Convert encoder's final states into decoder initial states.
        self.hidden_bridge = nn.Linear(hidden_size * 2, hidden_size)
        self.cell_bridge = nn.Linear(hidden_size * 2, hidden_size)

        # English token embeddings
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        # Attention layers
        self.encoder_attention = nn.Linear(
            hidden_size * 2, hidden_size
        )
        self.decoder_attention = nn.Linear(
            hidden_size, hidden_size
        )
        self.attention_score = nn.Linear(hidden_size, 1)

        # Decoder receives the word embedding + attention context.
        self.decoder = nn.LSTM(
            input_size=embedding_dim + hidden_size * 2,
            hidden_size=hidden_size,
            batch_first=True
        )

        # Predict the next English token.
        self.output_layer = nn.Linear(
            hidden_size + hidden_size * 2,
            vocab_size
        )

    def forward(self, poses, pose_lengths, target):

        batch_size = poses.size(0)

        # [B, Frames, 543, 3]
        # becomes [B, Frames, 1629]
        poses = poses.reshape(batch_size, poses.size(1), -1)

        # Project pose features
        poses = self.input_projection(poses)

        # Pack sequences so padding does not affect the encoder.
        packed = pack_padded_sequence(
            poses,
            pose_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, (hidden, cell) = self.encoder(packed)

        encoder_outputs, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=poses.size(1)
        )

        # Combine forward and backward encoder states.
        final_hidden = torch.cat(
            (hidden[-2], hidden[-1]), dim=1
        )
        final_cell = torch.cat(
            (cell[-2], cell[-1]), dim=1
        )

        decoder_hidden = torch.tanh(
            self.hidden_bridge(final_hidden)
        ).unsqueeze(0)

        decoder_cell = torch.tanh(
            self.cell_bridge(final_cell)
        ).unsqueeze(0)

        # Precompute encoder attention representations.
        encoder_features = self.encoder_attention(encoder_outputs)

        # Mask padded pose frames.
        frame_positions = torch.arange(
            encoder_outputs.size(1),
            device=poses.device
        ).unsqueeze(0)

        padding_mask = frame_positions >= pose_lengths.to(
            poses.device
        ).unsqueeze(1)

        # Teacher forcing: feed all target tokens except EOS.
        decoder_inputs = target[:, :-1]

        outputs = []

        for t in range(decoder_inputs.size(1)):

            # Embed the current English token.
            word_embedding = self.embedding(
                decoder_inputs[:, t]
            )

            # Use the previous decoder hidden state for attention.
            decoder_features = self.decoder_attention(
                decoder_hidden[-1]
            ).unsqueeze(1)

            scores = self.attention_score(
                torch.tanh(
                    encoder_features + decoder_features
                )
            ).squeeze(-1)

            scores = scores.masked_fill(
                padding_mask, float("-inf")
            )

            attention_weights = torch.softmax(scores, dim=1)

            # Weighted combination of encoder outputs.
            context = torch.bmm(
                attention_weights.unsqueeze(1),
                encoder_outputs
            ).squeeze(1)

            # Decoder input = current word + visual context.
            decoder_input = torch.cat(
                (word_embedding, context), dim=1
            ).unsqueeze(1)

            decoder_output, (decoder_hidden, decoder_cell) = (
                self.decoder(
                    decoder_input,
                    (decoder_hidden, decoder_cell)
                )
            )

            # Predict the next word.
            prediction = self.output_layer(
                torch.cat(
                    (decoder_output.squeeze(1), context),
                    dim=1
                )
            )

            outputs.append(prediction)

        return torch.stack(outputs, dim=1)


# Create the model
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = ISLTranslator(
    vocab_size=len(vocab),
    hidden_size=128,
    embedding_dim=128
).to(device)

print("Vocabulary size:", len(vocab))
print("Model device:", device)
print("Model parameters:", sum(p.numel() for p in model.parameters()))


Vocabulary size: 39084
Model device: cuda
Model parameters: 21241133


In [ ]:
import subprocess

result = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
)

print(result.stdout)

NVIDIA-SMI has failed because you do not have suffient permissions. Please try running as an administrator.



In [ ]:
import torch

if torch.cuda.is_available():

    free_memory, total_memory = torch.cuda.mem_get_info()

    print("GPU:", torch.cuda.get_device_name(0))
    print("Total memory:", round(total_memory / 1024**3, 2), "GB")
    print("Free memory:", round(free_memory / 1024**3, 2), "GB")
    print("Allocated by PyTorch:", round(
        torch.cuda.memory_allocated() / 1024**3, 2
    ), "GB")
    print("Reserved by PyTorch:", round(
        torch.cuda.memory_reserved() / 1024**3, 2
    ), "GB")

else:
    print("CUDA unavailable")

GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
Total memory: 6.0 GB
Free memory: 0.0 GB
Allocated by PyTorch: 0.0 GB
Reserved by PyTorch: 0.0 GB


In [ ]:
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

try:
    x = torch.randn(1024, 1024, device="cuda")
    print("GPU tensor created:", x.shape)

    torch.cuda.synchronize()
    print("GPU test successful!")

    del x
    torch.cuda.empty_cache()

except Exception as e:
    print("GPU test failed:", repr(e))

CUDA: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
GPU tensor created: torch.Size([1024, 1024])
GPU test successful!


In [ ]:
free, total = torch.cuda.mem_get_info()

print("Free VRAM:", round(free / 1024**3, 2), "GB")
print("Total VRAM:", round(total / 1024**3, 2), "GB")

Free VRAM: 5.03 GB
Total VRAM: 6.0 GB


In [97]:
device = torch.device("cuda")

model = ISLTranslator(
    vocab_size=len(vocab),
    hidden_size=128,
    embedding_dim=128
).to(device)

print("Model successfully loaded!")
print("Device:", next(model.parameters()).device)
print(
    "Parameters:",
    sum(p.numel() for p in model.parameters())
)

Model successfully loaded!
Device: cuda:0
Parameters: 21241133


In [98]:
names = [
    "train_data",
    "ds",
    "pose_ds",
    "tokenization",
    "create_vocab",
    "valid_indices",
    "vocab",
    "ISLDataset",
    "train_dataset",
    "collate_fn",
    "train_loader",
    "ISLTranslator"
]

for name in names:
    print(name, ":", name in globals())

train_data : True
ds : True
pose_ds : True
tokenization : True
create_vocab : True
valid_indices : True
vocab : True
ISLDataset : True
train_dataset : True
collate_fn : True
train_loader : True
ISLTranslator : True


In [99]:
import torch

free, total = torch.cuda.mem_get_info()

print("Free VRAM:", round(free / 1024**3, 2), "GB")
print("Total VRAM:", round(total / 1024**3, 2), "GB")

Free VRAM: 4.85 GB
Total VRAM: 6.0 GB


In [100]:
device = torch.device("cuda")

model = ISLTranslator(
    vocab_size=len(vocab),
    hidden_size=128,
    embedding_dim=128
).to(device)

print("Model loaded successfully!")
print("Device:", next(model.parameters()).device)
print("Parameters:", sum(p.numel() for p in model.parameters()))

Model loaded successfully!
Device: cuda:0
Parameters: 21241133


In [101]:
poses, texts, pose_lengths, text_lengths = next(iter(train_loader))

poses = poses.to(device)
texts = texts.to(device)
pose_lengths = pose_lengths.to(device)

model.eval()

with torch.no_grad():
    predictions = model(poses, pose_lengths, texts)

print("Pose input:", poses.shape)
print("Text input:", texts.shape)
print("Predictions:", predictions.shape)

poses/0000/1e9591008a0d-67.pose: reconstructing file:   0%|          |  0.00B /  675kB            

poses/0000/1e9591008a0d-67.pose: downloading bytes:           |  0.00B            

poses/0006/WcsKgEX-iKo--8.pose: reconstructing file:   0%|          |  0.00B / 1.53MB            

poses/0006/WcsKgEX-iKo--8.pose: downloading bytes:           |  0.00B            

poses/0011/xxTstiC8Jkc--11.pose: reconstructing file:   0%|          |  0.00B / 1.95MB            

poses/0011/xxTstiC8Jkc--11.pose: downloading bytes:           |  0.00B            

... (23 outputs left out)

poses/0003/D9O50mQ7vRo_e3.pose: downloading bytes:           |  0.00B            

poses/0009/j1NvRvKdrlA--11.pose: reconstructing file:   0%|          |  0.00B / 1.84MB            

poses/0009/j1NvRvKdrlA--11.pose: downloading bytes:           |  0.00B            

Pose input: torch.Size([16, 437, 543, 3])
Text input: torch.Size([16, 23])
Predictions: torch.Size([16, 22, 39084])


In [102]:
import torch.nn as nn

criterion = nn.CrossEntropyLoss(ignore_index=0)

In [103]:
import torch.optim as optim

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
num_epochs = 3

for epoch in range(num_epochs):

    model.train()
    total_loss = 0

    for batch_idx, (poses, texts, pose_lengths, text_lengths) in enumerate(train_loader):

        poses = poses.to(device)
        texts = texts.to(device)
        pose_lengths = pose_lengths.to(device)

        optimizer.zero_grad()

        predictions = model(poses, pose_lengths, texts)

        targets = texts[:, 1:]

        loss = criterion(
            predictions.reshape(-1, predictions.size(-1)),
            targets.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

        if (batch_idx + 1) % 100 == 0:
            print(
                f"Epoch [{epoch+1}/{num_epochs}] "
                f"Batch [{batch_idx+1}/{len(train_loader)}] "
                f"Loss: {loss.item():.4f}"
            )

    average_loss = total_loss / len(train_loader)

    print(f"\nEpoch {epoch+1} Average Loss: {average_loss:.4f}\n")

poses/0001/5vQOomiY-50--7.pose: reconstructing file:   0%|          |  0.00B / 3.48MB            

poses/0001/5vQOomiY-50--7.pose: downloading bytes:           |  0.00B            

poses/0011/vbuEvtq_3Fg--42.pose: reconstructing file:   0%|          |  0.00B / 4.78MB            

poses/0011/vbuEvtq_3Fg--42.pose: downloading bytes:           |  0.00B            

poses/0011/xm5YC-KDXyg--11.pose: reconstructing file:   0%|          |  0.00B / 2.32MB            

poses/0011/xm5YC-KDXyg--11.pose: downloading bytes:           |  0.00B            

... (327 outputs left out)

poses/0007/a5FzG5AIReo--14.pose: downloading bytes:           |  0.00B            

poses/0013/-BJZvEAOuUM--5.pose: reconstructing file:   0%|          |  0.00B / 1.76MB            

poses/0013/-BJZvEAOuUM--5.pose: downloading bytes:           |  0.00B            

: 

In [ ]:
import torch

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    free, total = torch.cuda.mem_get_info()

    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Allocated: {allocated:.2f} GB")
    print(f"Reserved: {reserved:.2f} GB")
    print(f"Free VRAM: {free / 1024**3:.2f} GB")
    print(f"Total VRAM: {total / 1024**3:.2f} GB")